# Mamba-UNet — Colab (TSR / Fourier, 128×128)

Обучение `segmentation/Mamba-UNet` на GPU Colab.

## Что загрузить (важно)

Сырые `.mat` (~21 GB) + кэш (~16 GB) в Colab через Upload **не тащи**. Используй **Google Drive**.

### Рекомендуемый набор на Drive (~16–17 GB)

На Mac (из корня репо):

```bash
cd /Users/user/Education/CVYandexCamp/thermal-control-ya-project

# папка на Drive (через Desktop Drive или rclone)
mkdir -p ~/Google\ Drive/My\ Drive/irt_colab

rsync -ah --progress artifacts/cache/ ~/Google\ Drive/My\ Drive/irt_colab/cache/
rsync -ah --progress labels/kaggle_binary_masks labels/tpu_binary_masks \
  ~/Google\ Drive/My\ Drive/irt_colab/labels/

# опционально — уже посчитанные фичи (~0.4 GB), ускорит первый эпох
rsync -ah --progress artifacts/features/ ~/Google\ Drive/My\ Drive/irt_colab/features/
```

Итоговая структура на Drive:

```text
MyDrive/irt_colab/
  cache/          # *.npy + index.json   ← обязательно
  labels/
    kaggle_binary_masks/
    tpu_binary_masks/
  features/       # опционально
```

`.mat` **не нужны**, если есть `cache/` + `index.json`. Ноутбук создаст пустые заглушки `data/*.mat`, чтобы yaml правильно привязал маски (kaggle vs tpu).

### Код репозитория

1. Запушь актуальный `segmentation/` + `irt_data/` на GitHub и сделай `git clone`, **или**
2. Заархивируй код без тяжёлых данных и положи zip на Drive:

```bash
cd /Users/user/Education/CVYandexCamp
zip -r ~/Google\ Drive/My\ Drive/irt_colab/thermal-code.zip thermal-control-ya-project \
  -x 'thermal-control-ya-project/data/*' \
  -x 'thermal-control-ya-project/artifacts/*' \
  -x 'thermal-control-ya-project/.git/*' \
  -x '*/__pycache__/*' \
  -x '*/*.tar' \
  -x '*/.mplconfig/*'
```

Runtime → **GPU** (T4 достаточно).

## 0. Конфиг

In [ ]:
# === правь под себя ===
DRIVE_IRT = "/content/drive/MyDrive/irt_colab"  # папка с cache/ labels/ [features/]
CODE_SOURCE = "zip"  # "zip" | "clone" | "already"

# если CODE_SOURCE == "zip":
DRIVE_CODE_ZIP = f"{DRIVE_IRT}/thermal-code.zip"

# если CODE_SOURCE == "clone":
GIT_URL = "https://github.com/tomatoCoderq/thermal-control-ya-project.git"
GIT_BRANCH = "main"  # или своя ветка с segmentation/

VARIANT = "tsr"       # "tsr" | "fourier"
EPOCHS = 50
BATCH_SIZE = 8        # T4: 4–8; A100 можно 16
SAMPLES_PER_VIDEO = 20
TEST_EVERY = 4
NUM_WORKERS = 2
LR = 3e-4
WEIGHT_DECAY = 1e-4

ROOT = "/content/thermal-control-ya-project"  # куда распакуем/клонируем код

## 1. Drive + код

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

assert Path(DRIVE_IRT).is_dir(), f"Нет папки {DRIVE_IRT} — сначала залей cache/labels на Drive"
assert (Path(DRIVE_IRT) / "cache" / "index.json").is_file(), "Нужен cache/index.json"
print("Drive OK:", DRIVE_IRT)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

root = Path(ROOT)

if CODE_SOURCE == "already":
    assert root.is_dir(), f"Нет {root}"
elif CODE_SOURCE == "clone":
    if root.exists():
        print("already cloned:", root)
    else:
        subprocess.check_call(
            ["git", "clone", "--branch", GIT_BRANCH, "--depth", "1", GIT_URL, str(root)]
        )
elif CODE_SOURCE == "zip":
    zpath = Path(DRIVE_CODE_ZIP)
    assert zpath.is_file(), f"Нет zip: {zpath}"
    if root.exists():
        shutil.rmtree(root)
    shutil.unpack_archive(str(zpath), "/content")
    # zip мог содержать thermal-control-ya-project/ ...
    if not root.is_dir():
        cands = list(Path("/content").glob("**/segmentation/Mamba-UNet/train.py"))
        assert cands, "В zip нет segmentation/Mamba-UNet"
        root = cands[0].parents[2]
        ROOT = str(root)
        print("ROOT detected:", ROOT)
else:
    raise ValueError(CODE_SOURCE)

assert (root / "segmentation" / "Mamba-UNet" / "train.py").is_file()
assert (root / "irt_data").is_dir()
print("code OK:", root)

## 2. Зависимости

In [ ]:
%pip install -q pydantic PyYAML albumentations opencv-python-headless scikit-learn matplotlib tqdm pillow scipy

import os

os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 3. Симлинки данных + заглушки `.mat`

Кэш и маски с Drive → в дерево репо. Пустые `data/*.mat` нужны только чтобы `discover_mat_files` правильно выбрал папку масок (kaggle vs tpu).

In [ ]:
import json
from pathlib import Path

root = Path(ROOT)
drive_irt = Path(DRIVE_IRT)

def link(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_symlink() or dst.exists():
        if dst.is_symlink() or dst.is_file():
            dst.unlink()
        else:
            # не трогаем чужие директории — перелинкуем через rename
            bak = dst.with_name(dst.name + ".bak")
            if bak.exists():
                import shutil

                shutil.rmtree(bak)
            dst.rename(bak)
    dst.symlink_to(src.resolve(), target_is_directory=src.is_dir())
    print(f"link {dst} -> {src}")

link(drive_irt / "cache", root / "artifacts" / "cache")

lab_dst = root / "labels"
lab_dst.mkdir(parents=True, exist_ok=True)
for name in ("kaggle_binary_masks", "tpu_binary_masks"):
    src = drive_irt / "labels" / name
    assert src.is_dir(), f"Нет {src}"
    link(src, lab_dst / name)

feat_src = drive_irt / "features"
if feat_src.is_dir():
    link(feat_src, root / "artifacts" / "features")
else:
    (root / "artifacts" / "features").mkdir(parents=True, exist_ok=True)
    print("features/ нет на Drive — построятся при первом проходе (медленнее)")

# заглушки .mat по id из кэша
index = json.loads((drive_irt / "cache" / "index.json").read_text())
vids = list(index["videos"].keys())
data_dir = root / "data"
data_dir.mkdir(parents=True, exist_ok=True)
for vid in vids:
    stub = data_dir / f"{vid}.mat"
    if not stub.exists():
        stub.write_bytes(b"")  # пустой файл — только для glob + mask routing
print(f"stubs: {len(vids)} files in {data_dir}")
print("sample ids:", vids[:5], "...", vids[-3:])

## 4. Проверка датасета

In [ ]:
import sys
from pathlib import Path

root = Path(ROOT)
seg = root / "segmentation"
mamba = seg / "Mamba-UNet"
unet = seg / "U-Net"
for p in (root, seg, mamba):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from irt_cfg import load_cfg
from common.data import build_train_test_loaders

yaml_path = unet / "dataset_tsr.yaml"
cfg = load_cfg(yaml_path, train=True)
cfg.samples_per_video = SAMPLES_PER_VIDEO

train_loader, test_loader, train_ds, test_ds = build_train_test_loaders(
    cfg,
    size=128,
    test_every=TEST_EVERY,
    batch_size=BATCH_SIZE,
    variant=VARIANT,
    num_workers=0,  # быстрая проверка
)
x, y = train_ds[0]
print(
    f"variant={VARIANT} | videos train/test={len(train_ds.video_ids)}/{len(test_ds.video_ids)}\n"
    f"samples train/test={len(train_ds)}/{len(test_ds)} | batch={BATCH_SIZE}\n"
    f"x={tuple(x.shape)} y={tuple(y.shape)} dtype={x.dtype}"
)
assert x.shape[0] == 6 and x.shape[-1] == 128

## 5. Train Mamba-UNet

In [ ]:
import sys
from pathlib import Path

import torch

root = Path(ROOT)
seg = root / "segmentation"
mamba_dir = seg / "Mamba-UNet"
unet = seg / "U-Net"
for p in (root, seg, mamba_dir):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from common.data import build_train_test_loaders
from common.device import get_device
from common.loop import run_epoch
from common.mps_train import optimize_model_mps
from common.tracking import HistoryTracker
from irt_cfg import load_cfg
from model import MambaUNet

device = get_device("cuda" if torch.cuda.is_available() else "cpu")
print("device=", device)

yaml_path = unet / "dataset_tsr.yaml"
cfg = load_cfg(yaml_path, train=True)
cfg.samples_per_video = SAMPLES_PER_VIDEO

train_loader, test_loader, train_ds, test_ds = build_train_test_loaders(
    cfg,
    size=128,
    test_every=TEST_EVERY,
    batch_size=BATCH_SIZE,
    variant=VARIANT,
    num_workers=NUM_WORKERS,
)
in_ch = int(train_ds[0][0].shape[0])
print(
    f"MambaUNet {VARIANT} | in_ch={in_ch} | bs={BATCH_SIZE} | {EPOCHS} ep\n"
    f"train {len(train_ds.video_ids)} vids / {len(train_ds)} samples | "
    f"test {len(test_ds.video_ids)} vids / {len(test_ds)} samples"
)

model = MambaUNet(in_channels=in_ch, num_classes=1)
model = optimize_model_mps(model, device, channels_last=False, compile_model=False)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

run_dir = mamba_dir / "runs" / VARIANT
tracker = HistoryTracker(run_dir, tag=f"mamba-{VARIANT}")
ckpt_best = mamba_dir / f"model_mamba_{VARIANT}_best.tar"
ckpt_last = mamba_dir / f"model_mamba_{VARIANT}_last.tar"

best_iou, best_epoch = -1.0, -1
for epoch in range(1, EPOCHS + 1):
    train_loss, train_m = run_epoch(
        model, train_loader, device, opt, desc=f"train {epoch}"
    )
    test_loss, test_m = run_epoch(model, test_loader, device, desc=f"test {epoch}")
    row = tracker.log(epoch, train_loss, test_loss, train_m, test_m)
    print(tracker.format_line(row, EPOCHS))

    if test_m["iou"] > best_iou:
        best_iou, best_epoch = test_m["iou"], epoch
        torch.save(model.state_dict(), ckpt_best)
        print(f"  >>> best test IoU {best_iou:.4f} @ epoch {best_epoch}")

    if device.type == "cuda":
        torch.cuda.empty_cache()

torch.save(model.state_dict(), ckpt_last)
print(f"done. best IoU {best_iou:.4f} @ {best_epoch}")
print("ckpt:", ckpt_best)
print("history:", tracker.json_path)

## 6. Кривые + сохранение на Drive

In [ ]:
import shutil
from pathlib import Path

from common.plot_history import load_history, plot_history

mamba_dir = Path(ROOT) / "segmentation" / "Mamba-UNet"
run_dir = mamba_dir / "runs" / VARIANT
curves = run_dir / "curves.png"
hist = load_history(run_dir)
plot_history(hist, title=f"mamba-{VARIANT}", save=curves, show=True)
print("saved", curves)

out = Path(DRIVE_IRT) / "runs_mamba" / VARIANT
out.mkdir(parents=True, exist_ok=True)
for p in [
    mamba_dir / f"model_mamba_{VARIANT}_best.tar",
    mamba_dir / f"model_mamba_{VARIANT}_last.tar",
    run_dir / "history.json",
    run_dir / "metrics.csv",
    curves,
]:
    if p.is_file():
        shutil.copy2(p, out / p.name)
        print("->", out / p.name)

## Если OOM

- `BATCH_SIZE = 4` или `2`
- `SAMPLES_PER_VIDEO = 10` (короче эпоха)
- Runtime → смена GPU / High-RAM

## Если нет `mamba_ssm`

В репо уже есть pure-PyTorch selective scan — на CUDA Colab работает без установки `mamba_ssm` (чуть медленнее родных ядер, для обучения ок).